# Baseline

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import umap.umap_ as umap
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
# 2: DBSCAN кластеризация

# Загрузка эмбеддингов
embeddings = np.load("embeddings_smart.npy")

# Стандартизация
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

# Применяем DBSCAN с фиксированными параметрами
# (выберите eps и min_samples по вашему усмотрению)
eps = 0.5
min_samples = 10
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
labels = dbscan.fit_predict(embeddings_scaled)

# Добавление меток кластеров в датасет
df["dbscan_cluster"] = labels

# Функция для расчёта Жаккаровской дистанции (межкластерное расстояние)
def calculate_jaccard_distance(labels, embeddings):
    cluster_distances = {}
    unique_labels = set(labels) - {-1}  # Игнорируем шум (-1)

    for label in unique_labels:
        cluster_points = embeddings[labels == label]
        if len(cluster_points) > 1:
            # Среднее расстояние между всеми парами точек в кластере
            intra_cluster_dist = np.mean(pairwise_distances(cluster_points))
            cluster_distances[label] = intra_cluster_dist
        else:
            cluster_distances[label] = 0.0

    # Среднее расстояние между кластерами (межкластерное расстояние)
    inter_cluster_distances = []
    for label1 in unique_labels:
        for label2 in unique_labels:
            if label1 < label2:
                points1 = embeddings[labels == label1]
                points2 = embeddings[labels == label2]
                dist = np.mean(pairwise_distances(points1, points2))
                inter_cluster_distances.append(dist)

    avg_inter_cluster_dist = np.mean(inter_cluster_distances) if inter_cluster_distances else 0.0

    return cluster_distances, avg_inter_cluster_dist

# Расчёт метрик
cluster_distances, avg_inter_cluster_dist = calculate_jaccard_distance(labels, embeddings_scaled)

# Вывод результатов
print("\nРезультаты DBSCAN:")
print(f"Всего кластеров: {len(cluster_distances)}")
print(f"Среднее внутрикластерное расстояние: {np.mean(list(cluster_distances.values())):.4f}")
print(f"Среднее межкластерное расстояние: {avg_inter_cluster_dist:.4f}")
print("\nРасстояние внутри каждого кластера:")
for cluster, dist in cluster_distances.items():
    print(f"Кластер {cluster}: {dist:.4f}")

# Сохранение результатов
df.to_csv("articles_without_orcid_dbscan.csv", index=False)
print("\nРезультаты DBSCAN сохранены в articles_without_orcid_dbscan.csv")